# SVM practice: linear support vector machine on a simple dataset

Before using our cancer data, this notebook practises the basics the professor listed:
1. What `fit()` and `predict()` do
2. The main setting, `C`
3. Why we standardize features with `StandardScaler`
4. How a `Pipeline` combines scaling and the model

We use scikit-learn's built-in **breast cancer** dataset: 569 tumours, 30 measurements each, labelled malignant or benign. It is small and clean, so it's good for learning.

In [7]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix

data = load_breast_cancer()
X, y = data.data, data.target          # X = measurements, y = label (1 = benign, 0 = malignant)
print("Samples, features:", X.shape)
print("Label counts:", pd.Series(y).value_counts().to_dict())

Samples, features: (569, 30)
Label counts: {1: 357, 0: 212}


## 1. Split into training and test data

The model learns from the **training** set and is checked on the **test** set it has never seen. This tells us how well it would work on new patients.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=0)
print("Training samples:", len(X_train), " Test samples:", len(X_test))

Training samples: 426  Test samples: 143


## 2. `fit()` and `predict()`

- `fit(X_train, y_train)`: **training**. The SVM looks at the training points and finds the straight line (in 30 dimensions, a flat "hyperplane") with the widest gap between the two classes.
- `predict(X_test)`: **prediction**. For each new sample it checks which side of that line the sample falls on and returns the class.

In [9]:
svm = SVC(kernel="linear", C=1.0)
svm.fit(X_train, y_train)          # learn the boundary
pred = svm.predict(X_test)         # use the boundary on new data

print("First 10 predictions:", pred[:10])
print("First 10 true labels:", y_test[:10])
print("Test accuracy:", round(accuracy_score(y_test, pred), 3))

First 10 predictions: [1 0 0 1 0 1 0 0 0 1]
First 10 true labels: [1 0 0 1 0 1 0 0 0 1]
Test accuracy: 0.944


## 3. Why standardize? (`StandardScaler`)

The features are on very different scales (some around 0.01, others in the thousands). An SVM measures distances, so large-number features would dominate. `StandardScaler` rescales every feature to mean 0 and standard deviation 1.

**Important:** the scaler must learn the mean and standard deviation from the **training data only**, then apply them to the test data.

In [10]:
print("Feature ranges before scaling:")
print("  smallest feature max:", X_train.max(axis=0).min().round(3))
print("  largest feature max: ", X_train.max(axis=0).max().round(1))

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # learn mean/std from training data, then scale
X_test_s = scaler.transform(X_test)         # reuse the SAME mean/std on the test data

svm_scaled = SVC(kernel="linear", C=1.0).fit(X_train_s, y_train)
print("Accuracy without scaling:", round(accuracy_score(y_test, pred), 3))
print("Accuracy with scaling:   ", round(accuracy_score(y_test, svm_scaled.predict(X_test_s)), 3))

Feature ranges before scaling:
  smallest feature max: 0.022
  largest feature max:  4254.0
Accuracy without scaling: 0.944
Accuracy with scaling:    0.965


## 4. `Pipeline`: scaling + model in one object

Doing the scaling by hand is easy to get wrong. A `Pipeline` chains the steps, so `fit()` scales using the training data and then trains the SVM, and `predict()` applies the same scaling before predicting. This is how we will use it on the cancer data, because it keeps the test data out of the scaling inside cross-validation automatically.

In [11]:
pipe = Pipeline([
    ("scale", StandardScaler()),
    ("svm", SVC(kernel="linear", C=1.0)),
])
pipe.fit(X_train, y_train)
pipe_pred = pipe.predict(X_test)
print("Pipeline test accuracy:", round(accuracy_score(y_test, pipe_pred), 3))
print("Confusion matrix (rows = true, columns = predicted):")
print(confusion_matrix(y_test, pipe_pred))

Pipeline test accuracy: 0.965
Confusion matrix (rows = true, columns = predicted):
[[50  3]
 [ 2 88]]


## 5. The main setting: `C`

`C` controls how strict the SVM is about mistakes on the training data:
- **Small C** (for example 0.01): allows some training mistakes for a wider, smoother gap. Simpler model, less overfitting.
- **Large C** (for example 100): tries hard to classify every training point correctly. Can overfit.

Here we compare a few values with 5-fold cross-validation (train on 4 parts, test on the 5th, repeat).

In [12]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
for C in [0.001, 0.01, 0.1, 1, 10, 100]:
    p = Pipeline([("scale", StandardScaler()), ("svm", SVC(kernel="linear", C=C))])
    scores = cross_val_score(p, X, y, cv=cv, scoring="accuracy")
    print(f"C = {C:<6} mean accuracy = {scores.mean():.3f}  (folds: {', '.join(f'{s:.2f}' for s in scores)})")

C = 0.001  mean accuracy = 0.939  (folds: 0.93, 0.96, 0.92, 0.93, 0.95)
C = 0.01   mean accuracy = 0.970  (folds: 0.95, 0.99, 0.96, 0.98, 0.96)
C = 0.1    mean accuracy = 0.977  (folds: 0.96, 0.98, 0.97, 0.99, 0.98)
C = 1      mean accuracy = 0.975  (folds: 0.96, 0.97, 0.98, 0.98, 0.98)
C = 10     mean accuracy = 0.963  (folds: 0.96, 0.96, 0.97, 0.95, 0.97)
C = 100    mean accuracy = 0.961  (folds: 0.96, 0.96, 0.96, 0.96, 0.96)


## What to remember

| Idea | Meaning |
|---|---|
| `fit()` | Learn the boundary from training data |
| `predict()` | Use the boundary to label new data |
| Linear kernel | The boundary is a straight line (a flat hyperplane) |
| `C` | Smaller = simpler, more forgiving boundary. Larger = stricter, can overfit |
| `StandardScaler` | Puts features on the same scale; learn it from training data only |
| `Pipeline` | Chains scaling + model so the scaling is always done correctly |

For our project we start with the **fixed** setting `C = 1.0`, as the professor asked. Tuning comes later.